# CLI walkthrough

This tutorial runs scPertEval end-to-end from the command line on a tiny synthetic dataset.
It is a runnable notebook (use the download/Colab buttons above), and the outputs shown here
were produced by running it.

In [1]:
# In Colab or a fresh environment, install scPertEval first:
# %pip install scperteval

For a real benchmark you would point the CLI at a preprocessed `.h5ad` — see
[Datasets](../../user-guide/datasets.md) for the available preprocessed datasets and how to download them.

## Set up a dataset

We build a 60-gene toy dataset with a control population and four perturbations, each with
its own distinct block of up-regulated genes, and save it as an `.h5ad`:

In [2]:
import pathlib
import tempfile

import anndata as ad
import numpy as np

rng = np.random.default_rng(0)
ng, n_ctrl, n_pert = 60, 150, 120
de_genes = {"pertA": range(0, 6), "pertB": range(15, 21), "pertC": range(30, 36), "pertD": range(45, 51)}

parts = [rng.poisson(1.0, (n_ctrl, ng)).astype(np.float32)]
labels = ["control"] * n_ctrl
for name, genes in de_genes.items():
    x = rng.poisson(1.0, (n_pert, ng)).astype(np.float32)
    x[:, list(genes)] += 6.0  # up-regulate this perturbation's marker genes
    parts.append(x)
    labels += [name] * n_pert

adata = ad.AnnData(np.vstack(parts))
adata.var_names = [f"g{i}" for i in range(ng)]
adata.obs["perturbation"] = labels

workdir = pathlib.Path(tempfile.mkdtemp())
data_path = workdir / "toy.h5ad"
adata.write_h5ad(data_path)

print(adata)
print("\nperturbations:", adata.obs["perturbation"].value_counts().to_dict())

AnnData object with n_obs × n_vars = 630 × 60
    obs: 'perturbation'

perturbations: {'control': 150, 'pertA': 120, 'pertB': 120, 'pertC': 120, 'pertD': 120}


## Calibrate protocols against built-in controls

`calibrate` scores each protocol against a positive and a negative control and reports a
calibrated **DRF** (or **BDS**) value — a measure of whether the protocol can tell real
perturbation signal from an uninformative baseline. The CLI and the Python entry point
take the same arguments; both are shown below and produce the same result.

In [3]:
%%bash -s "$data_path" "$workdir"
scperteval calibrate $1 -p pearson_ctrl,mse --out-dir $2


toy · t-test · subsample=8192 · seed=42 · output=drf

protocol                   representation

 space          mean    median
---------------------------------------------------------------------

--
pearson_ctrl               centroid       full          0.923     0.921
mse                      

  centroid       full          0.821     0.820

-> /tmp/tmpnbntn516/toy__2026-07-03T160142__drf.csv


In [4]:
from scperteval.cli import main

main(["calibrate", str(data_path), "-p", "pearson_ctrl,mse", "--out-dir", str(workdir)])


toy · t-test · subsample=8192 · seed=42 · output=drf

protocol                   representation space          mean    median
-----------------------------------------------------------------------
pearson_ctrl               centroid       full          0.923     0.921
mse                        centroid       full          0.821     0.820

-> /tmp/tmpnbntn516/toy__2026-07-03T160143__drf.csv


Each run also writes a per-perturbation CSV. Let's load it:

In [5]:
import pandas as pd

drf_csv = next(workdir.glob("*__drf.csv"))
pd.read_csv(drf_csv).head()

,protocol,perturbation,raw_positive,raw_negative,drf,dataset,de_method,subsample,seed
0,pearson_ctrl,pertA,0.901133,-0.223609,0.919200,toy,t-test,8192,42
1,pearson_ctrl,pertB,0.914060,-0.213915,0.929203,toy,t-test,8192,42
2,pearson_ctrl,pertC,0.900720,-0.222744,0.918805,toy,t-test,8192,42
3,pearson_ctrl,pertD,0.907469,-0.210240,0.923542,toy,t-test,8192,42
4,mse,pertA,0.901117,4.996488,0.819650,toy,t-test,8192,42


## Export differential expression

The `de` command runs the ground-truth differential-expression step on its own and writes
the per-gene results to HDF5 — handy for inspecting the signal a protocol is scored against.
Again, the CLI and Python forms are equivalent:

In [ ]:
%%bash -s "$data_path" "$workdir"
scperteval de $1 --methods t-test --out-dir $2

/nfs/research/saezrodriguez/hakem/project/metrics-review/scPertEval/src/scperteval/context.py:224: U

a large share of the dataset, so its leave-one-out reference is much smaller than other perturbation

s'. Raise --subsample (or expect noisier single-cell metrics for this perturbation).
  keep = self.r

eference().keep(pert)


/nfs/research/saezrodriguez/hakem/project/metrics-review/scPertEval/src/scperteval/context.py:224: U

...

/scPertEval/src/scperteval/context.py:224: UserWarning: Excluding 'pertB' removes 120/480 (25%) of t

he reference sample -- this perturbation is a large share of the dataset, so its leave-one-out refer

ence is much smaller than other perturbations'. Raise --subsample (or expect noisier single-cell met

rics for this perturbation).
  keep = self.reference().keep(pert)


-> /tmp/tmpnbntn516/toy__2026-07-03T160144__de.h5  (4 perturbations, methods=['t-test'])


In [7]:
main(["de", str(data_path), "--methods", "t-test", "--out-dir", str(workdir)])
[p.name for p in workdir.glob("*__de.h5")]

-> /tmp/tmpnbntn516/toy__2026-07-03T160144__de.h5  (4 perturbations, methods=['t-test'])


/nfs/research/saezrodriguez/hakem/project/metrics-review/scPertEval/src/scperteval/context.py:224: UserWarning: Excluding 'pertA' removes 120/480 (25%) of the reference sample -- this perturbation is a large share of the dataset, so its leave-one-out reference is much smaller than other perturbations'. Raise --subsample (or expect noisier single-cell metrics for this perturbation).
  keep = self.reference().keep(pert)
/nfs/research/saezrodriguez/hakem/project/metrics-review/scPertEval/src/scperteval/context.py:224: UserWarning: Excluding 'pertD' removes 120/480 (25%) of the reference sample -- this perturbation is a large share of the dataset, so its leave-one-out reference is much smaller than other perturbations'. Raise --subsample (or expect noisier single-cell metrics for this perturbation).
  keep = self.reference().keep(pert)
/nfs/research/saezrodriguez/hakem/project/metrics-review/scPertEval/src/scperteval/context.py:224: UserWarning: Excluding 'pertC' removes 120/480 (25%) of t

['toy__2026-07-03T160144__de.h5']

## Where to next

- **Score predictions against ground truth** with `scperteval score dataset.h5ad predictions.h5ad` — see [Scoring predictions](../../user-guide/scoring.md).
- List the available building blocks with `scperteval list protocols` (also `de-methods`, `spaces`, `sources`, `calibrators`).
- Read [Calibration](../../user-guide/calibration.md) for what DRF and BDS actually measure.